In [1]:
import os

class Graph:
    def __init__(self, n):
        self.maxNodes = n
        self.edgeList = []
        self.adj_list = []
        for i in range(0, n + 1):
            self.adj_list.append([])

    def add_edge(self, u, v):
        self.adj_list[u].append(v)
        self.adj_list[v].append(u)

        self.edgeList.append((u,v))


def read_edges_from_file(filename):
    with open(filename, 'r') as file:
        edges = [tuple(map(int, line.strip().split())) for line in file]
    return edges[0][0], edges[1:]

def create_graph_from_edges(edges, n):
    graph = Graph(n)
    for u, v in edges:
        graph.add_edge(u, v)
    return graph

In [2]:
from pyomo.environ import *
import networkx as nx

def ILP_Solver_Accurate(V, k, dist):
    model = ConcreteModel()

    # Variables
    vi = [(v, i) for v in V for i in range(1, k + 1)]
    model.x = Var(vi, domain=Binary, initialize=0)
    model.z = Var(domain=NonNegativeReals, bounds=(0, k), initialize=0)

    # Objective function
    model.Objective = Objective(expr=model.z, sense=minimize)

    # Constraints
    # Fractional coloring constraint
    model.unique_coloring = ConstraintList()
    for v in V:
        model.unique_coloring.add(sum(model.x[(v, i)] for i in range(1, k + 1)) == 1)

    # Adjacent coloring constraint
    model.adjacent_coloring = ConstraintList()
    for v in V:
        for u in V:
            for i in range(1, k + 1):
                if dist[u][v] <= i and u != v:
                    model.adjacent_coloring.add(model.x[(v, i)] + model.x[(u, i)] <= 1)

    # Coloring constraint
    model.coloring_constraint = ConstraintList()
    for v in V:
        for i in range(1, k + 1):
            model.coloring_constraint.add(i * model.x[(v, i)] <= model.z)

    # Solve
    solverFactory = SolverFactory('glpk')
    solverFactory._version_timeout = 600
    results = solverFactory.solve(model)
    
    # Return results
    return model.x, model.z, model, results

In [3]:
def get_distance_matrix_lol(g : Graph):
    G = nx.Graph()
    G.add_edges_from(g.edgeList)
    return nx.floyd_warshall(G)


In [4]:
def read_graphs_from_file():
    folder_path = '../generatedgraphs'  # Update this with the path to your folder
    graph_files = [file for file in os.listdir(folder_path) if file.endswith('.txt')]

    graphs = []
    for file in graph_files:
        n, edges = read_edges_from_file(os.path.join(folder_path, file))
        graph = create_graph_from_edges(edges, n)  # Replace n with the appropriate value
        graphs.append(graph)

    return graphs

In [5]:
graphs = read_graphs_from_file()

In [6]:
len(graphs)

30

In [8]:
n, edges = read_edges_from_file(os.path.join("../generatedgraphs", "56nodes.txt"))

In [9]:
def ILP_Solver_Approximate(V, k, dist):
    model = ConcreteModel()

    # Variables
    vi = [(v, i) for v in V for i in range(1, k + 1)]
    model.x = Var(vi, domain=NonNegativeReals, initialize=0, bounds=(0, 1))
    model.z = Var(domain=NonNegativeReals, bounds=(0, k), initialize=0)

    # Objective function
    model.Objective = Objective(expr=model.z, sense=minimize)

    # Constraints
    # Fractional coloring constraint
    model.unique_coloring = ConstraintList()
    for v in V:
        model.unique_coloring.add(sum(model.x[(v, i)] for i in range(1, k + 1)) == 1)

    # Adjacent coloring constraint
    model.adjacent_coloring = ConstraintList()
    for v in V:
        for u in V:
            for i in range(1, k + 1):
                if dist[u][v] <= i and u != v:
                    model.adjacent_coloring.add(model.x[(v, i)] + model.x[(u, i)] <= 1)

    # Coloring constraint
    model.coloring_constraint = ConstraintList()
    for v in V:
        for i in range(1, k + 1):
            model.coloring_constraint.add(i * model.x[(v, i)] <= model.z)

    # Solve
    solverFactory = SolverFactory('glpk')
    solverFactory._version_timeout = 600
    results = solverFactory.solve(model)
    
    # Return results
    return model.x, model.z, model, results

In [10]:
g56 = create_graph_from_edges(edges, n)

In [12]:
V = [i for i in range(1, g56.maxNodes + 1)]
k = g56.maxNodes
mat = get_distance_matrix_lol(g56)
x, z, problem, results = ILP_Solver_Approximate(V, k, mat)

In [13]:
print(f"Objective value (z): {z()}\n")

Objective value (z): 0.216850622451335



In [16]:
results.write()

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 0.216850622451335
  Upper bound: 0.216850622451335
  Number of objectives: 1
  Number of constraints: 160868
  Number of variables: 3137
  Number of nonzeros: 324760
  Sense: minimize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 0
      Number of created subproblems: 0
  Error rc: 0
  Time: 10.50039291381836
# ----------------------------------------------------------
#   Solution Information
# --

In [53]:
from tqdm import tqdm
import pandas as pd
import csv

fieldnames = ['Nodes', 'Number of colors used', 'File Name']
count: int = 1

for graph in tqdm(graphs):

    V = [i for i in range(1, graph.maxNodes + 1)]
    k = graph.maxNodes
    mat = get_distance_matrix_lol(graph)
    x, z, problem, results = ILP_Solver_Accurate(V, k, mat)
    file_name = f"status_graph_id_{count}.txt"


    with open('output.csv', 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if csvfile.tell() == 0:
            writer.writeheader()
        writer.writerow({'Nodes': graph.maxNodes, 'Number of colors used': z(), 'File Name': file_name})


    with open(file_name, "w") as file:
        file.write(f"Objective value (z): {z()}\n")

        file.write(f"Proper Coloring:\n")
        coloring = {}

        for idx in x:
            if x[idx]() == 1:
                file.write(f"node → {idx[0]}: color → {idx[1]}\n")

        json_repr = results.json_repn()
        
        file.write("\n\nProblem:\n")
        for key, value in json_repr["Problem"][0].items():
            file.write(f"{key}: {value}\n")
        file.write("\nSolver:\n")
        for key, value in json_repr["Solver"][0].items():
            if isinstance(value, dict):
                file.write(key + ":\n")
                for k, v in value.items():
                    file.write(f"  {k}: {v}\n")
            else:
                file.write(f"{key}: {value}\n")
        file.write("\nSolution:\n")
        for key, value in json_repr["Solution"][0].items():
            file.write(f"{key}: {value}\n")

    count += 1

100%|██████████| 30/30 [22:45<00:00, 45.52s/it]


In [55]:
from rich.console import Console

console = Console()

df = pd.read_csv('output.csv')
console.print(df)

Nodes  Number of colors used               File Name
0      20                    4.0   status_graph_id_1.txt
1      20                    4.0   status_graph_id_2.txt
2      28                    4.0   status_graph_id_3.txt
3      29                    4.0   status_graph_id_4.txt
4      22                    5.0   status_graph_id_5.txt
5      20                    4.0   status_graph_id_6.txt
6      20                    5.0   status_graph_id_7.txt
7      25                    4.0   status_graph_id_8.txt
8      20                    4.0   status_graph_id_9.txt
9      30                    5.0  status_graph_id_10.txt
10     25                    5.0  status_graph_id_11.txt
11     21                    4.0  status_graph_id_12.txt
12     26                    4.0  status_graph_id_13.txt
13     28                    4.0  status_graph_id_14.txt
14     20                    4.0  status_graph_id_15.txt
15     27                    4.0  status_graph_id_16.txt
16     27                    5.0  status_graph_id_17.txt
17     22                    4.0  status_graph_id_18.txt
18     29                    5.0  status_graph_id_19.txt
19     30                    5.0  status_graph_id_20.txt
20     20                    4.0  status_graph_id_21.txt
21     29                    5.0  status_graph_id_22.txt
22     24                    4.0  status_graph_id_23.txt
23     26                    4.0  status_graph_id_24.txt
24     28                    5.0  status_graph_id_25.txt
25     29                    4.0  status_graph_id_26.txt
26     24                    5.0  status_graph_id_27.txt
27     30                    5.0  status_graph_id_28.txt
28     23                    5.0  status_graph_id_29.txt
29     27                    4.0  status_graph_id_30.txt

In [ ]:
d4b_small = pd.read_csv("../stastistics/stats.csv", sep=',', index_col=False)
d4b_small["leaves/ratio"] = d4b_small["Total Leaves"] / d4b_small["Nodes"] * 100

d4b_small

In [67]:
with open("edges.txt", "w") as f:
    f.write(f"{len(graphs)}\n")

for graph in graphs:
    with open("edges.txt", "a") as f:
        f.write(str(graph.maxNodes) + " " + str(len(graph.edgeList)) + "\n")
        edges = graph.edgeList
        for edge in edges:
            f.write(f"{edge[0]} {edge[1]}\n")